# Kapitel 3 — Embeddings

Jetzt haben wir für jedes Wort eine Zahl. Aber eine Zahl wie 9246 ist
nur ein Label. Das Modell kann aus Labels nicht lernen. Es braucht
Bedeutung. Es muss wissen, dass cat und dog ähnlich sind und dass
cat und car unterschiedlich sind.

Ein Embedding ist ein Vektor aus Gleitkommazahlen, der die Bedeutung
eines Wortes erfasst. Man kann es sich so vorstellen, als würde man
jedem Wort eine Koordinate in einem großen Raum geben. Wörter mit
ähnlicher Bedeutung landen nahe beieinander. Wörter mit unterschiedlicher Bedeutung landen weit auseinander.

Diese Embeddings starten als Zufallszahlen. Während des Trainings
lernt das Modell, verwandte Wörter näher zusammenzurücken. Nach
ausreichendem Training werden cat und dog Nachbarn sein, während car
auf der anderen Seite des Raums liegt. Dieses Notebook zeigt dir, wie
diese Lookup-Tabelle funktioniert und wie die Magie des Lernens sie
bedeutungsvoll macht.

## Importe

In [1]:
import torch
import torch.nn as nn
import math

## Der Embedding-Layer

In [2]:
class Embedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        return self.embed(x)

## Eine Embedding-Tabelle erstellen und testen

In [3]:
vocab_size = 1000
d_model = 768

embedding = Embedding(vocab_size, d_model)
print(f"Embedding table shape: [{vocab_size}, {d_model}]")
print(f"Total parameters: {sum(p.numel() for p in embedding.parameters()):,}")
print()

token_ids = torch.tensor([[12, 45, 678, 2, 890]])
print(f"Input token IDs: {token_ids}")
print(f"Input shape: {token_ids.shape}")
print()

output = embedding(token_ids)
print(f"Output shape: {output.shape}")
print(f"Each token is now a {d_model}-dimensional vector")
print(f"First token vector preview: {output[0, 0, :5].tolist()} ...")

Embedding table shape: [1000, 768]
Total parameters: 768,000

Input token IDs: tensor([[ 12,  45, 678,   2, 890]])
Input shape: torch.Size([1, 5])

Output shape: torch.Size([1, 5, 768])
Each token is now a 768-dimensional vector
First token vector preview: [-16.388792037963867, -13.410012245178223, 8.839740753173828, 23.09480094909668, -17.038358688354492] ...


## Der Skalierungs-Trick

Wir verwenden RoPE für die Positionscodierung. RoPE rotiert Vektoren,
anstatt Positionsinformationen zu addieren. Rotation erhält die
Vektor-Magnitude, sodass keine Skalierung nötig ist. LLaMA verwendet
denselben Ansatz.
auf eine gute Skala, bevor wir später die Positionscodierung hinzufügen.
Ohne diese Skalierung würden die Positionsinformationen verloren gehen.

In [4]:
# Embeddings funktionieren mit oder ohne Skalierung, wenn RoPE verwendet wird.
# LLaMA wendet keine Skalierung an, daher folgen wir dieser Konvention.
embed_raw = nn.Embedding(vocab_size, d_model)
x = embed_raw(token_ids)
print(f"Embedding values: mean={x.mean():.4f}, std={x.std():.4f}")
print(f"Shape: {x.shape}")


Without scaling: mean=-0.0146, std=1.0058
With scaling: mean=-0.4043, std=27.8724


## Embeddings für unseren Tokenizer nachschlagen

Jetzt verbinden wir das mit dem echten Tokenizer aus Kapitel 2.

In [5]:
from dataclasses import dataclass
import tiktoken

@dataclass
class TokenizerConfig:
    name: str = "gpt2"
    vocab_size: int = 50257

class SimpleTokenizer:
    def __init__(self, config=None):
        self.config = config or TokenizerConfig()
        self.enc = tiktoken.get_encoding(self.config.name)
        self.eos_token = "<|endoftext|>"
        self.eos_token_id = self.enc.encode(
            self.eos_token, allowed_special={self.eos_token}
        )[0]

    def encode(self, text):
        return self.enc.encode(text, allowed_special={self.eos_token})

    def decode(self, ids):
        return self.enc.decode(ids)

    @property
    def vocab_size(self):
        return self.config.vocab_size

tokenizer = SimpleTokenizer()
text = "The cat sat on the mat"
token_ids = tokenizer.encode(text)
print(f"Text: {text}")
print(f"Token IDs: {token_ids}")
print()

real_embedding = Embedding(tokenizer.vocab_size, d_model=768)
input_tensor = torch.tensor([token_ids])
vectors = real_embedding(input_tensor)
print(f"Output shape: {vectors.shape}")
print(f"Each of the {len(token_ids)} tokens is now a 768-dim vector")
print(f"The word cat (token {token_ids[1]}) has vector: {vectors[0, 1, :3].tolist()} ...")

Text: The cat sat on the mat
Token IDs: [464, 3797, 3332, 319, 262, 2603]

Output shape: torch.Size([1, 6, 768])
Each of the 6 tokens is now a 768-dim vector
The word cat (token 3797) has vector: [-43.199928283691406, -31.866304397583008, 17.82335090637207] ...
